# はじめに

このノートでは、PyTorch Geometricを利用してグラフニューラルネットワークを構築することを目指す。最初はPyTorchの基礎からはじめ、グラフニューラルネットワーク、PyTorch Geometricと内容を進めていく。

今回は、PyTorch Geometricのドキュメントの内容を実行しながら理解を深める。最後におまけ程度にトランスダクティブ学習についてまとめておく。

- [Advanced Mini-Batching — pytorch_geometric documentation](https://pytorch-geometric.readthedocs.io/en/2.5.2/advanced/batching.html)

下記はグラフニューラルネットワークを理解するための参考サイト。

- [グラフニューラルネットワーク | 佐藤 竜馬](https://www.amazon.co.jp/%E3%82%B0%E3%83%A9%E3%83%95%E3%83%8B%E3%83%A5%E3%83%BC%E3%83%A9%E3%83%AB%E3%83%8D%E3%83%83%E3%83%88%E3%83%AF%E3%83%BC%E3%82%AF-%E6%A9%9F%E6%A2%B0%E5%AD%A6%E7%BF%92%E3%83%97%E3%83%AD%E3%83%95%E3%82%A7%E3%83%83%E3%82%B7%E3%83%A7%E3%83%8A%E3%83%AB%E3%82%B7%E3%83%AA%E3%83%BC%E3%82%BA-%E4%BD%90%E8%97%A4-%E7%AB%9C%E9%A6%AC/dp/4065347823)
- [グラフ深層学習のすゝめ。 - YouTube](https://www.youtube.com/watch?v=7rgXi3Xp6NI)
- [GCN — グラフ道場](https://yuya-s.github.io/GraphDojo/01GCN.html)
- [Tutorial 6: Basics of Graph Neural Networks](https://lightning.ai/docs/pytorch/stable/notebooks/course_UvA-DL/06-graph-neural-networks.html)
- [Static and Dynamic Attention: Implications for Graph Neural Networks](https://medium.com/data-science/static-and-dynamic-attention-implications-for-graph-neural-networks-eda0d9d7b60a)
- [CS224W | Home](https://web.stanford.edu/class/cs224w/index.html)
- [nn.labml.ai/ja/graphs](https://nn.labml.ai/#:~:text=%E2%9C%A8%20Graph%20Neural%20Networks)
- [Understanding Convolutions on Graphs](https://distill.pub/2021/understanding-gnns/)
- [A Gentle Introduction to Graph Neural Networks](https://distill.pub/2021/gnn-intro/)

## ミニバッチ


ミニバッチ処理では、サンプルを1つずつ処理するのではなく、データを統一された表現にグループ化し、効率的に並列処理できるようにする。この新しく追加された次元の長さは、ミニバッチに含まれるサンプル数と同じであり、それを`batch_size`と呼ぶ。例えば、1枚の画像`(3,224,224)`があったとすると、50枚の画像をまとめると`(50,3,224,224)`となる。このとき、50がバッチサイズとなる。グラフでは、この方法は使えない。グラフは任意の数のノードやエッジを保持するデータ構造を持つため、上記のアプローチは実現不可能であるか、大量の不要なメモリ消費を引き起こす。

Pytorch Geometricでは、隣接行列を対角線状に積み重ね（複数の独立したサブグラフを含む巨大なグラフを作成する）、ノードとターゲットの特徴量をノード次元で単純に連結することでミニバッチを実現する。

$$
\begin{split}\mathbf{A} = \begin{bmatrix} \mathbf{A}_1 & & \\ & \ddots & \\ & & \mathbf{A}_n \end{bmatrix}, \qquad \mathbf{X} = \begin{bmatrix} \mathbf{X}_1 \\ \vdots \\ \mathbf{X}_n \end{bmatrix}, \qquad \mathbf{Y} = \begin{bmatrix} \mathbf{Y}_1 \\ \vdots \\ \mathbf{Y}_n \end{bmatrix}.\end{split}
$$

このようにミニバッチを構成できる背景には、GNNの計算はメッセージパッシングを基本とするため、異なるグラフに属する2つのノード間でメッセージを交換をする必要がない。隣接行列のブロックがずれるので下記の通り、異なるグラフの情報が交じることはない。また、Pytorchの隣接関係はエッジのみ(非0要素)を保持するため、追加のオーバーヘッドも発生しない。

In [ ]:
import polars as pl
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


# ----- Graph 1 -----
A1 = torch.tensor([[1.0, 1.0], [1.0, 1.0]])

X1 = torch.tensor([[1.0], [2.0]])
H1 = A1 @ X1
print(f"H1 = A1 @ X1: \n{H1}")

# ----- Graph 2 -----
A2 = torch.tensor([[1.0, 0.0], [0.0, 1.0]])

X2 = torch.tensor([[10.0], [20.0]])
H2 = A2 @ X2
print(f"H2 = A2 @ X2: \n{H2}")


# ----- Block Diagonal Adjacency -----
A = torch.block_diag(A1, A2)

# ----- Concatenate Node Features -----
X = torch.cat([X1, X2], dim=0)

print("A =\n", A)
print("X =\n", X)

# ----- Message Passing (GCN without weight) -----
H = A @ X

print("H =\n", H)

H1 = A1 @ X1: 
tensor([[3.],
        [3.]])
H2 = A2 @ X2: 
tensor([[10.],
        [20.]])
A =
 tensor([[1., 1., 0., 0.],
        [1., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])
X =
 tensor([[ 1.],
        [ 2.],
        [10.],
        [20.]])
H =
 tensor([[ 3.],
        [ 3.],
        [10.],
        [20.]])


Pytorch Geometricでは、`Dataloder`によって、複数のグラフを1つのグラフにまとめることできる。内部では、`__inc__`と`__cat_dim__`関数をもとにどの方向につながるのか、どれくらいずらすのかを管理し、ミニバッチを作成する。

In [ ]:
edge_index = [[0, 1], [1, 2]]


class MyData:

    def __init__(self, x, edge_index):
        self.x = x
        self.edge_index = edge_index
        self.num_nodes = x.size(0)

    # どれだけ増やすか？
    def __inc__(self, key, value):
        if "index" in key:
            return self.num_nodes
        else:
            return 0

    # どの次元で連結するか？
    def __cat_dim__(self, key, value):
        if "index" in key:
            return 1
        else:
            return 0


def simple_collate(data_list):
    batch = {}
    cumulative_nodes = 0

    for i, data in enumerate(data_list):
        print(f"\n--- Processing Graph {i} ---")

        for key in ["x", "edge_index"]:
            value = getattr(data, key)

            cat_dim = data.__cat_dim__(key, value)
            inc = data.__inc__(key, value)

            print(f"{key}:")
            print(" original:\n", value)
            print(" cat_dim:", cat_dim)
            print(" inc:", inc)

            if key == "edge_index":
                value = value + cumulative_nodes
                print(" shifted:\n", value)

            if key not in batch:
                batch[key] = value
            else:
                batch[key] = torch.cat([batch[key], value], dim=cat_dim)

        cumulative_nodes += data.num_nodes
        print(" cumulative_nodes:", cumulative_nodes)

    return batch

In [ ]:
# Graph1
x1 = torch.tensor([[1.0], [2.0], [3.0]])
edge_index1 = torch.tensor([[0, 1], [1, 2]])

# Graph2
x2 = torch.tensor([[10.0], [20.0], [30.0]])
edge_index2 = torch.tensor([[0, 1], [1, 2]])

data1 = MyData(x1, edge_index1)
data2 = MyData(x2, edge_index2)

batch = simple_collate([data1, data2])

print("\n=== Final Batch ===")
print("x:\n", batch["x"])
print("edge_index:\n", batch["edge_index"])


--- Processing Graph 0 ---
x:
 original:
 tensor([[1.],
        [2.],
        [3.]])
 cat_dim: 0
 inc: 0
edge_index:
 original:
 tensor([[0, 1],
        [1, 2]])
 cat_dim: 1
 inc: 3
 shifted:
 tensor([[0, 1],
        [1, 2]])
 cumulative_nodes: 3

--- Processing Graph 1 ---
x:
 original:
 tensor([[10.],
        [20.],
        [30.]])
 cat_dim: 0
 inc: 0
edge_index:
 original:
 tensor([[0, 1],
        [1, 2]])
 cat_dim: 1
 inc: 3
 shifted:
 tensor([[3, 4],
        [4, 5]])
 cumulative_nodes: 6

=== Final Batch ===
x:
 tensor([[ 1.],
        [ 2.],
        [ 3.],
        [10.],
        [20.],
        [30.]])
edge_index:
 tensor([[0, 1, 3, 4],
        [1, 2, 4, 5]])


エッジの接続情報の部分を次に考える必要がある。バッチとして複数のグラフを1つの行列にまとめた際に、ノードが持つ接続情報もずらす必要がある。例えば、2つのグラフ$\mathcal{G_s}$と$\mathcal{G_t}$があったとする。同じ大きな行列として扱うために、$\mathcal{G_t}$の接続情報は、$\mathcal{G_s}$の接続情報の数分スライドさせる必要がある。
Pytorch Geometricでは`DataLoader`クラスを利用すればうまくスライドしてくれる。

In [ ]:
# グラフ1（3頂点）
x1 = torch.tensor([[1.0, 0.0], [2.0, 0.0], [3.0, 0.0]])
edge_index1 = torch.tensor([[0, 1, 2], [1, 2, 0]])
data1 = Data(x=x1, edge_index=edge_index1)

# グラフ2（2頂点）
x2 = torch.tensor([[10.0, 0.0], [20.0, 0.0]])
edge_index2 = torch.tensor([[0], [1]])
data2 = Data(x=x2, edge_index=edge_index2)

# DataLoaderでMini-batch
loader = DataLoader([data1, data2], batch_size=2, follow_batch=["x1", "x2"])

batch = next(iter(loader))
print("Batch:", batch)
print("x:\n", batch.x)
print("edge_index:\n", batch.edge_index)
print("batch_index:\n", batch.batch)

Batch: DataBatch(x=[5, 2], edge_index=[2, 4], batch=[5], ptr=[3])
x:
 tensor([[ 1.,  0.],
        [ 2.,  0.],
        [ 3.,  0.],
        [10.,  0.],
        [20.,  0.]])
edge_index:
 tensor([[0, 1, 2, 3],
        [1, 2, 0, 4]])
batch_index:
 tensor([0, 0, 0, 1, 1])


## ボートレースとミニバッチ

最後に、競艇の着順予測モデルの実践において、ミニバッチを利用する場合の例をまとめておく。まずはサンプルデータを生成する。5レースで特徴量が3つというサンプルデータを生成する。

In [ ]:
np.random.seed(1989)
rows = []
for race_id in range(5):  # レース
    features = np.random.randn(6, 3)  # 6艇 × 3特徴量
    ranks = np.random.permutation([1, 2, 3, 4, 5, 6])

    for boat in range(6):
        rows.append(
            {
                "race_id": "race" + str(race_id),
                "boat": boat + 1,
                "x1": features[boat, 0],
                "x2": features[boat, 1],
                "x3": features[boat, 2],
                "y": ranks[boat],
            }
        )

df = pl.DataFrame(rows)
df = df.sort(["race_id", "boat"])

pl.Config(tbl_rows=-1)
df

race_id,boat,x1,x2,x3,y
str,i64,f64,f64,f64,i64
"""race0""",1,-0.263718,0.08532,0.430007,4
"""race0""",2,0.895069,0.556956,0.441123,3
"""race0""",3,0.375388,-0.149947,0.776825,5
"""race0""",4,-0.022542,1.605578,-0.367356,1
"""race0""",5,0.355542,0.169827,2.518024,2
"""race0""",6,0.139225,1.159444,0.587207,6
"""race1""",1,0.753399,0.571694,0.378271,3
"""race1""",2,-1.705966,-0.244093,0.355013,5
"""race1""",3,-1.181625,0.811026,0.583242,6


エッジ情報を作成する。ここでは完全グラフとして、エッジ情報を作成する。また、エッジ属性は艇番差として、エッジ情報も使わないが作成しておく。

In [ ]:
def complete_graph_with_edge_attr(num_nodes):
    edges = []
    # edge_attrs = []

    for i in range(num_nodes):
        for j in range(num_nodes):
            if i != j:
                edges.append([i, j])

                # 例：艇番差
                # edge_attrs.append([abs(i - j)])

    edge_index = torch.tensor(edges).t().contiguous()
    # edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

    return edge_index  # , edge_attr


## edge_attrは使わないがイメージとして記載。
edge_index = complete_graph_with_edge_attr(6)
edge_index.T

tensor([[0, 1],
        [0, 2],
        [0, 3],
        [0, 4],
        [0, 5],
        [1, 0],
        [1, 2],
        [1, 3],
        [1, 4],
        [1, 5],
        [2, 0],
        [2, 1],
        [2, 3],
        [2, 4],
        [2, 5],
        [3, 0],
        [3, 1],
        [3, 2],
        [3, 4],
        [3, 5],
        [4, 0],
        [4, 1],
        [4, 2],
        [4, 3],
        [4, 5],
        [5, 0],
        [5, 1],
        [5, 2],
        [5, 3],
        [5, 4]])

`Data`クラスを利用してレースごとに情報をまとめる。

In [20]:
data_list = []

# polars group_by returns a tuple (group_key, group_df)
for _, group in df.group_by("race_id"):

    x = torch.tensor(group.select(["x1", "x2", "x3"]).to_numpy(), dtype=torch.float)
    y = torch.tensor(group["y"].to_numpy(), dtype=torch.long)
    edge_index, edge_attr = complete_graph_with_edge_attr(6)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data_list.append(data)

data_list

[Data(x=[6, 3], edge_index=[30], edge_attr=[30], y=[6]),
 Data(x=[6, 3], edge_index=[30], edge_attr=[30], y=[6]),
 Data(x=[6, 3], edge_index=[30], edge_attr=[30], y=[6]),
 Data(x=[6, 3], edge_index=[30], edge_attr=[30], y=[6]),
 Data(x=[6, 3], edge_index=[30], edge_attr=[30], y=[6])]

`DataLoader`クラスでバッチを作成。ここではバッチサイズを2として、バッチを作成する。

In [ ]:
loader = DataLoader(data_list, batch_size=2, shuffle=False)

for batch in loader:
    print("x shape:", batch.x.shape)
    print("edge_index shape:", batch.edge_index.shape)
    print("edge_attr shape:", batch.edge_attr.shape)
    print("y shape:", batch.y.shape)
    print("batch vector shape:", batch.batch.shape)
    print("batch vector:", batch.batch)
    print("-" * 40)

x shape: torch.Size([12, 3])
edge_index shape: torch.Size([60])
edge_attr shape: torch.Size([60])
y shape: torch.Size([12])
batch vector shape: torch.Size([12])
batch vector: tensor([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1])
----------------------------------------
x shape: torch.Size([12, 3])
edge_index shape: torch.Size([60])
edge_attr shape: torch.Size([60])
y shape: torch.Size([12])
batch vector shape: torch.Size([12])
batch vector: tensor([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1])
----------------------------------------
x shape: torch.Size([6, 3])
edge_index shape: torch.Size([30])
edge_attr shape: torch.Size([30])
y shape: torch.Size([6])
batch vector shape: torch.Size([6])
batch vector: tensor([0, 0, 0, 0, 0, 0])
----------------------------------------


`edge_index`の部分がわかりにくい。本来`0-index`ではあるが、グラフ1で`[0,1,2,3,4,5]`を使用しているので、オフセットされて、グラフ2からは6始まりのインデックスで管理さる。batchベクトルはどこまでが1つのグラフなのかを管理している。

In [ ]:
b = next(iter(loader))
print("[X]", "-" * 40)
print(b.x)
print("[edge_index]", "-" * 40)
print(b.edge_index)
print("[edge_attr]", "-" * 40)
print(b.edge_attr)
print("[y]", "-" * 40)
print(b.y)
print("[batch]", "-" * 40)
print(b.batch)
print("-" * 40)

[X] ----------------------------------------
tensor([[-0.2637,  0.0853,  0.4300],
        [ 0.8951,  0.5570,  0.4411],
        [ 0.3754, -0.1499,  0.7768],
        [-0.0225,  1.6056, -0.3674],
        [ 0.3555,  0.1698,  2.5180],
        [ 0.1392,  1.1594,  0.5872],
        [ 0.7534,  0.5717,  0.3783],
        [-1.7060, -0.2441,  0.3550],
        [-1.1816,  0.8110,  0.5832],
        [ 2.4490,  0.9207, -0.3776],
        [ 1.1051, -0.9443, -0.6151],
        [-1.7993, -0.4398,  0.9690]])
[edge_index] ----------------------------------------
tensor([ 0,  0,  0,  0,  0,  1,  1,  1,  1,  1,  2,  2,  2,  2,  2,  3,  3,  3,
         3,  3,  4,  4,  4,  4,  4,  5,  5,  5,  5,  5,  6,  6,  6,  6,  6,  7,
         7,  7,  7,  7,  8,  8,  8,  8,  8,  9,  9,  9,  9,  9, 10, 10, 10, 10,
        10, 11, 11, 11, 11, 11])
[edge_attr] ----------------------------------------
tensor([1, 2, 3, 4, 5, 0, 2, 3, 4, 5, 0, 1, 3, 4, 5, 0, 1, 2, 4, 5, 0, 1, 2, 3,
        5, 0, 1, 2, 3, 4, 1, 2, 3, 4, 5, 0, 2, 3, 

## トランスダクティブ学習

グラフニューラルネットワークを学んでいると、帰納学習(Inductive learning)と推移学習(Transductive learning)という言葉に出くわすことが多い。この違いについて簡単にまとめる。グラフニューラルネットワークでは、帰納学習と推移学習という2つの考え方がある。

帰納学習は、グラフやノードからラベルを予測する一般的なルールを学び、そのルールを学習時に見ていない新しいデータへ適用する方法である。

一方、推移学習は、1つの大きなグラフの中でラベル付きノードとラベルなしノードを同時に扱い、既知のラベルを手がかりに未知ノードのラベルを予測する方法である。つまり、帰納学習は新しいデータへの一般化を重視し、推移学習は同じグラフ内の未ラベル部分の補完を重視する。

<div align="center"><img src="./Transductive.png" width="600"></div>